# Week 9 — Day 2: Serving the Model with FastAPI

| Field | Value |
|:------|:------|
| **Phase** | Phase 3 — Deep Learning & Applied Project |
| **Sprint** | Sprint 4 (Week 9) — *Deployment & Production* |
| **Day** | Day 2 of 5 — Serving the Model with FastAPI |
| **Project** | Arabic Sentiment Classification |
| **Final model** | TF-IDF (10K features) + Logistic Regression (C=1.0) |
| **Test macro F1** | 0.8623 |
| **Notebook** | `BinX_Week_09/Day2/FastAPI.ipynb` |

> This notebook implements the **Day 2 lab**: serving the serialized Sprint 4 model
> through a local FastAPI REST API, with Pydantic validation, preprocessing reuse,
> and full notebook-vs-API consistency verification.

---

## 1. Day 2 Objectives

From the official Week 9 curriculum, Day 2 must deliver:

1. **FastAPI application** — local model-serving API
2. **Pydantic validation** — request schema enforcement
3. **Preprocessing reuse** — exact same pipeline as training
4. **Serialized artifact loading** — model + vectorizer + lemma table + config
5. **`/predict` POST endpoint** — raw text → JSON prediction
6. **Error handling** — graceful responses for invalid inputs
7. **`/docs` testing** — Swagger UI walkthrough
8. **Notebook vs API verification** — same input, same output
9. **Day 3 handoff** — document Streamlit integration requirements

---

## 2. Previous Work Context

### Final Model (from Week 8 / Day 1)

| Property | Value |
|:---------|:------|
| **Task** | Binary Arabic sentiment (Negative=0, Positive=1) |
| **Model** | Logistic Regression (C=1.0, max_iter=1000, random_state=42) |
| **Representation** | TF-IDF (10K features, min_df=2, sublinear_tf=True) |
| **Test Accuracy** | 0.8623 |
| **Test Macro F1** | 0.8623 |
| **Test ROC-AUC** | 0.9417 |

### Serialized Artifacts (from Day 1)

| Artifact | Type | Size |
|:---------|:-----|:-----|
| `artifacts/model.joblib` | LogisticRegression | ~79 KB |
| `artifacts/vectorizer.joblib` | TfidfVectorizer | ~386 KB |
| `artifacts/lemma_table.json` | dict (20K+ entries) | ~597 KB |
| `artifacts/preprocessing_config.json` | dict | ~2 KB |

### Day 1 Verification

- 10-sample known prediction: **all match**
- 3,000-sample full test set: **identical predictions and probabilities**
- Max probability difference: < 1e-6

---

## 3. Inference Pipeline

The complete inference flow (verified from Week 8 Day 1/4/5 and Day 1):

```
Raw Arabic Text
       ↓
normalize_text()          — Tashkeel removal, Alef unification, punctuation cleanup
       ↓
word_tokenize()           — NLTK Arabic tokenizer
       ↓
Filter digits/Latin       — Remove non-Arabic tokens
       ↓
unify_alef()              — Alef/Hamza/ta-marbuta normalization
       ↓
Protect negations         — Preserve sentiment-critical tokens
       ↓
Lemmatize                 — qalsadi dictionary-based Arabic lemmatization
       ↓
Remove stopwords          — NLTK Arabic stopwords (with negation protection)
       ↓
TF-IDF vectorizer         — Transform using fitted 10K-feature vocabulary
       ↓
Logistic Regression       — Binary classification
       ↓
Label mapping             — 0 → Negative, 1 → Positive
```

### Key design decisions

1. **Preprocessing is code-based** — deterministic functions, no learned state
2. **Only learned artifacts need serialization** — model, vectorizer, lemma table
3. **The preprocessing module is shared** — `preprocessing.py` ensures training/serving consistency
4. **Artifacts are loaded once at startup** — not on every request

---

## 4. Environment & Imports

In [1]:
import json, os, sys, time, warnings, re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

import joblib

import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk.tokenize import word_tokenize

try:
    import qalsadi.lemmatizer
    HAS_QALSADI = True
except ImportError:
    HAS_QALSADI = False

warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Add Day2 to path so preprocessing module is importable
sys.path.insert(0, str(Path.cwd()))
from preprocessing import (
    normalize_text, unify_alef, preprocess_text,
    load_artifacts, get_artifacts_dir,
    DIGIT_RE, LATIN_RE, NEGATION_WORDS, INTENSIFIER_WORDS,
    PROTECTED, STOP_WORDS
)

print('=' * 62)
print('ENVIRONMENT')
print('=' * 62)
print(f'  Python        : {sys.version.split()[0]}')
print(f'  NumPy         : {np.__version__}')
print(f'  scikit-learn  : {__import__("sklearn").__version__}')
print(f'  joblib        : {joblib.__version__}')
print(f'  qalsadi       : {"available" if HAS_QALSADI else "MISSING"}')
print(f'  SEED          : {SEED}')
print('=' * 62)
print('\n✓ Environment ready.')

ENVIRONMENT
  Python        : 3.13.15
  NumPy         : 2.5.1
  scikit-learn  : 1.9.0
  joblib        : 1.5.3
  qalsadi       : available
  SEED          : 42

✓ Environment ready.


---

## 5. Artifact Loading

Load all serialized artifacts from Day 1 using the shared `load_artifacts()` function.
This replicates the exact loading procedure verified in Day 1 Section 12.

In [2]:
artifacts = load_artifacts()

model = artifacts['model']
vectorizer = artifacts['vectorizer']
lemma_table = artifacts['lemma_table']
config = artifacts['config']
label_names = artifacts['label_names']
label_mapping = artifacts['label_mapping']

print('Loaded serialized artifacts:')
print(f'  Model       : {type(model).__name__}, C={model.C}')
print(f'  Vectorizer  : {type(vectorizer).__name__}, vocab={len(vectorizer.vocabulary_):,}')
print(f'  Lemma table : {len(lemma_table):,} entries')
print(f'  Config      : version {config["artifacts_version"]}')
print(f'  Label names : {label_names}')
print(f'  Label map   : {label_mapping}')
print('\n✓ All artifacts loaded successfully.')

Loaded serialized artifacts:
  Model       : LogisticRegression, C=1.0
  Vectorizer  : TfidfVectorizer, vocab=10,000
  Lemma table : 20,381 entries
  Config      : version 1.0.0
  Label names : ['Negative (0)', 'Positive (1)']
  Label map   : {0: 'Negative', 1: 'Positive'}

✓ All artifacts loaded successfully.


---

## 6. Preprocessing — Shared Module Verification

The preprocessing functions are imported from `preprocessing.py` — the same module
used by `main.py`. This guarantees training/serving consistency.

Below we verify the imported functions produce identical results to the Day 1 pipeline.

In [3]:
# Verify the imported functions match Day 1 behavior
test_raw = 'هذا المنتج ممتاز جداً وأنصح بشرائه للجميع'

cleaned = preprocess_text(test_raw, lemma_table=lemma_table)
features = vectorizer.transform([cleaned])
pred = model.predict(features)[0]
proba = model.predict_proba(features)[0]

print('Preprocessing verification (shared module):')
print(f'  Input : {test_raw}')
print(f'  Cleaned: {cleaned}')
print(f'  Features shape: {features.shape}')
print(f'  Prediction: {pred} ({label_names[pred]})')
print(f'  Probabilities: {label_names[0]}={proba[0]:.4f}, {label_names[1]}={proba[1]:.4f}')
assert pred == 1, f'Expected Positive, got {pred}'
print('\n✓ Preprocessing pipeline verified.')

Preprocessing verification (shared module):
  Input : هذا المنتج ممتاز جداً وأنصح بشرائه للجميع
  Cleaned: منتج ممتاز جدا انصاح بشير
  Features shape: (1, 10000)
  Prediction: 1 (Positive (1))
  Probabilities: Negative (0)=0.0239, Positive (1)=0.9761

✓ Preprocessing pipeline verified.


---

## 7. Pydantic Schema

The request schema enforces:
- `text` must be a string
- `text` must not be empty or whitespace-only

FastAPI/Pydantic rejects invalid requests before they reach the model.

In [4]:
from pydantic import BaseModel, Field, field_validator
from fastapi import HTTPException

class PredictionRequest(BaseModel):
    """Request schema for /predict endpoint."""
    text: str = Field(
        ...,
        min_length=1,
        description='Arabic review text to classify',
        json_schema_extra={'example': 'هذا المنتج ممتاز جدا'},
    )

    @field_validator('text')
    @classmethod
    def text_must_not_be_empty(cls, v):
        if not v.strip():
            raise ValueError('Text must not be empty or whitespace-only')
        return v

class PredictionResponse(BaseModel):
    """Response schema for /predict endpoint."""
    prediction: int = Field(description='Predicted class label (0 or 1)')
    label: str = Field(description='Human-readable sentiment label')
    confidence: float = Field(description='Confidence score for predicted class')
    probabilities: dict = Field(description='Probability for each class')
    model_version: str = Field(description='Artifacts version from config')
    preprocessing_steps: int = Field(description='Number of preprocessing steps applied')

print('Pydantic schemas defined:')
print(f'  PredictionRequest fields: {list(PredictionRequest.model_fields.keys())}')
print(f'  PredictionResponse fields: {list(PredictionResponse.model_fields.keys())}')

# Verify valid request works
req = PredictionRequest(text='هذا المنتج ممتاز جدا')
print(f'  Valid request: text={req.text!r}')

# Verify empty text is rejected
try:
    PredictionRequest(text='')
    print('  ERROR: Empty text was accepted!')
except Exception as e:
    print(f'  Empty text rejected: {type(e).__name__}')

# Verify whitespace-only is rejected
try:
    PredictionRequest(text='   ')
    print('  ERROR: Whitespace-only text was accepted!')
except Exception as e:
    print(f'  Whitespace-only rejected: {type(e).__name__}')

# Verify missing field is rejected
try:
    PredictionRequest()
    print('  ERROR: Missing text was accepted!')
except Exception as e:
    print(f'  Missing text rejected: {type(e).__name__}')

# Verify wrong type is rejected
try:
    PredictionRequest(text=12345)
    print('  ERROR: Wrong type was accepted!')
except Exception as e:
    print(f'  Wrong type rejected: {type(e).__name__}')

print('\n✓ Pydantic validation working correctly.')

Pydantic schemas defined:
  PredictionRequest fields: ['text']
  PredictionResponse fields: ['prediction', 'label', 'confidence', 'probabilities', 'model_version', 'preprocessing_steps']
  Valid request: text='هذا المنتج ممتاز جدا'
  Empty text rejected: ValidationError
  Whitespace-only rejected: ValidationError
  Missing text rejected: ValidationError
  Wrong type rejected: ValidationError

✓ Pydantic validation working correctly.


---

## 8. FastAPI Application

The application loads artifacts once at startup and exposes:
- `GET /` — health/root endpoint
- `GET /health` — lightweight health check
- `POST /predict` — classification endpoint

In [5]:
from fastapi import FastAPI

app = FastAPI(
    title='Arabic Sentiment Classification API',
    description='Serve Arabic sentiment predictions using TF-IDF + Logistic Regression.',
    version=config['artifacts_version'],
)

# Store loaded artifacts for endpoint use
app.state.model = model
app.state.vectorizer = vectorizer
app.state.lemma_table = lemma_table
app.state.config = config
app.state.label_names = label_names

print(f'FastAPI app created: {app.title} v{app.version}')

FastAPI app created: Arabic Sentiment Classification API v1.0.0


---

## 9. `/predict` Endpoint

In [6]:
@app.get('/')
def root():
    """Health/root endpoint confirming the API is running."""
    return {
        'status': 'ok',
        'message': 'Arabic Sentiment Classification API',
        'model_loaded': True,
        'model_type': type(app.state.model).__name__,
        'artifacts_version': app.state.config['artifacts_version'],
    }


@app.get('/health')
def health():
    """Lightweight health check."""
    return {'status': 'healthy'}


@app.post('/predict', response_model=PredictionResponse)
def predict_endpoint(request: PredictionRequest):
    """
    Classify the sentiment of an Arabic review.

    Accepts raw Arabic text, applies the exact same preprocessing pipeline
    used during training, and returns the predicted sentiment with
    confidence scores.
    """
    try:
        # Step 1: Preprocess (exact same function as training)
        cleaned = preprocess_text(request.text, lemma_table=app.state.lemma_table)

        # Step 2: Vectorize (loaded fitted TF-IDF vectorizer)
        features = app.state.vectorizer.transform([cleaned])

        # Step 3: Predict
        prediction = int(app.state.model.predict(features)[0])

        # Step 4: Get probabilities
        proba = app.state.model.predict_proba(features)[0]
        label = app.state.label_names[prediction]
        probabilities = {
            app.state.label_names[i]: round(float(proba[i]), 4)
            for i in range(2)
        }
        confidence = round(float(proba[prediction]), 4)

    except Exception as e:
        raise HTTPException(status_code=500, detail=f'Prediction failed: {str(e)}')

    return PredictionResponse(
        prediction=prediction,
        label=label,
        confidence=confidence,
        probabilities=probabilities,
        model_version=app.state.config['artifacts_version'],
        preprocessing_steps=len(app.state.config['preprocessing_steps']),
    )

print('Endpoints registered:')
for route in app.routes:
    if hasattr(route, 'methods'):
        print(f'  {route.methods} {route.path}')

Endpoints registered:
  {'HEAD', 'GET'} /openapi.json
  {'HEAD', 'GET'} /docs
  {'HEAD', 'GET'} /docs/oauth2-redirect
  {'HEAD', 'GET'} /redoc
  {'GET'} /
  {'GET'} /health
  {'POST'} /predict


---

## 10. Local Server Instructions

### Start the server

From the `BinX_Week_09/Day2/` directory:

```bash
uvicorn main:app --reload
```

Or run `main.py` directly:

```bash
python main.py
```

### Open the documentation

Navigate to:

```
http://127.0.0.1:8000/docs
```

### Test via `/docs`

1. Open the Swagger UI at `/docs`
2. Expand the `POST /predict` section
3. Click **Try it out**
4. Enter a request body:
   ```json
   {"text": "هذا المنتج ممتاز جدا"}
   ```
5. Click **Execute**
6. Inspect the JSON response

### Key endpoints

| Method | Path | Description |
|:-------|:-----|:------------|
| GET | `/` | Root/health — confirms API is running |
| GET | `/health` | Lightweight health check |
| POST | `/predict` | Classify Arabic sentiment |
| GET | `/docs` | Swagger UI documentation |

---

## 11. Valid Input Tests

Test the API with multiple meaningful Arabic sentiment examples.

In [7]:
from fastapi.testclient import TestClient

client = TestClient(app)

# Test cases: Arabic reviews with expected sentiment direction
test_cases = [
    {'text': 'هذا المنتج ممتاز جدا', 'desc': 'Positive - simple'},
    {'text': 'المنتج سيء جدا ولا أنصح به', 'desc': 'Negative - simple'},
    {'text': 'جودة عالية وشحن سريع', 'desc': 'Positive - short'},
    {'text': 'خدمة سيئة جدا وانتظار طويل', 'desc': 'Negative - short'},
    {'text': 'منتج عادي لا يستحق السعر', 'desc': 'Negative - moderate'},
    {'text': 'أحب هذا المتجر ومنتجاتهم ممتازة جدا', 'desc': 'Positive - moderate'},
    {'text': ' flewed but the delivery was quick', 'desc': 'Latin text (should handle gracefully)'},
]

print('=' * 70)
print('VALID INPUT TESTS')
print('=' * 70)

valid_results = []
for i, tc in enumerate(test_cases):
    resp = client.post('/predict', json={'text': tc['text']})
    assert resp.status_code == 200, f'Request {i+1} failed: {resp.status_code}'
    data = resp.json()
    valid_results.append({
        'desc': tc['desc'],
        'text': tc['text'][:30] + ('...' if len(tc['text']) > 30 else ''),
        'prediction': data['prediction'],
        'label': data['label'],
        'confidence': data['confidence'],
        'status': 'OK',
    })
    print(f'  {i+1}. {tc["desc"]}')
    print(f'     Text: {tc["text"][:40]}...')
    print(f'     → {data["label"]} (confidence={data["confidence"]:.4f})')
    print()

print(f'✓ All {len(test_cases)} valid input tests passed.')

VALID INPUT TESTS
  1. Positive - simple
     Text: هذا المنتج ممتاز جدا...
     → Positive (1) (confidence=0.9761)

  2. Negative - simple
     Text: المنتج سيء جدا ولا أنصح به...
     → Negative (0) (confidence=0.5063)

  3. Positive - short
     Text: جودة عالية وشحن سريع...
     → Positive (1) (confidence=0.7413)

  4. Negative - short
     Text: خدمة سيئة جدا وانتظار طويل...
     → Negative (0) (confidence=0.6725)

  5. Negative - moderate
     Text: منتج عادي لا يستحق السعر...
     → Negative (0) (confidence=0.7072)

  6. Positive - moderate
     Text: أحب هذا المتجر ومنتجاتهم ممتازة جدا...
     → Positive (1) (confidence=0.9678)

  7. Latin text (should handle gracefully)
     Text:  flewed but the delivery was quick...
     → Positive (1) (confidence=0.6714)

✓ All 7 valid input tests passed.


---

## 12. Invalid Input Tests

Verify that Pydantic validation correctly rejects invalid requests.

In [8]:
print('=' * 70)
print('INVALID INPUT TESTS')
print('=' * 70)

invalid_cases = [
    ({}, 'Missing text field'),
    ({'text': ''}, 'Empty text'),
    ({'text': '   '}, 'Whitespace-only text'),
    ({'text': 12345}, 'Wrong type (integer)'),
    ({'text': None}, 'None value'),
]

all_rejected = True
for i, (payload, desc) in enumerate(invalid_cases):
    resp = client.post('/predict', json=payload)
    rejected = resp.status_code != 200
    status = '✓ REJECTED' if rejected else '✗ ACCEPTED (should be rejected)'
    if not rejected:
        all_rejected = False
    print(f'  {i+1}. {desc}: {status} (HTTP {resp.status_code})')

print()
if all_rejected:
    print('✓ All invalid inputs correctly rejected.')
else:
    print('✗ Some invalid inputs were not rejected — investigate.')

INVALID INPUT TESTS
  1. Missing text field: ✓ REJECTED (HTTP 422)
  2. Empty text: ✓ REJECTED (HTTP 422)
  3. Whitespace-only text: ✓ REJECTED (HTTP 422)
  4. Wrong type (integer): ✓ REJECTED (HTTP 422)
  5. None value: ✓ REJECTED (HTTP 422)

✓ All invalid inputs correctly rejected.


---

## 13. Notebook vs FastAPI Verification

The **critical Day 2 serving correctness test**:

```
Same input
     ↓
Notebook pipeline  →  Prediction A

Same input
     ↓
FastAPI pipeline   →  Prediction B

A == B  ← MUST be true
```

In [9]:
def notebook_predict(raw_text, return_probs=False):
    """Prediction using the loaded artifacts (same as notebook inference)."""
    cleaned = preprocess_text(raw_text, lemma_table=lemma_table)
    features = vectorizer.transform([cleaned])
    prediction = int(model.predict(features)[0])
    label_name = label_names[prediction]
    if return_probs:
        probs = model.predict_proba(features)[0]
        return prediction, label_name, {label_names[i]: float(probs[i]) for i in range(2)}
    return prediction, label_name


def api_predict(raw_text):
    """Prediction via the FastAPI test client."""
    resp = client.post('/predict', json={'text': raw_text})
    assert resp.status_code == 200, f'API returned {resp.status_code}'
    return resp.json()


# Comparison test cases
comparison_samples = [
    'هذا المنتج ممتاز جدا',                        # Positive - simple
    'المنتج سيء جدا ولا أنصح به',                  # Negative - simple
    'جودة عالية وشحن سريع وخدمة ممتازة',          # Positive - longer
    'خدمة سيئة جدا وانتظار طويل ولا أنصح',        # Negative - longer
    'أحب هذا المتجر ومنتجاتهم ممتازة جدا',        # Positive - moderate
    'منتج عادي لا يستحق السعر المطلوب',           # Negative - moderate
    'هذا أفضل منتج استخدمته هذا العام',             # Positive
    'المنتج وصل تالف ومكسور',                     # Negative
]

print('=' * 70)
print('NOTEBOOK vs FASTAPI COMPARISON')
print('=' * 70)
print(f'{"Sample":<45s} {"Notebook":<12s} {"FastAPI":<12s} {"Match":<6s}')
print('-' * 70)

all_match = True
results_table = []

for i, sample in enumerate(comparison_samples):
    # Notebook prediction
    nb_pred, nb_label, nb_probs = notebook_predict(sample, return_probs=True)

    # API prediction
    api_data = api_predict(sample)
    api_pred = api_data['prediction']
    api_label = api_data['label']
    api_probs = {k: v for k, v in api_data['probabilities'].items()}

    # Compare predictions
    pred_match = (nb_pred == api_pred)

    # Compare probabilities (tolerance for floating-point)
    prob_match = all(
        abs(nb_probs[k] - api_probs[k]) < 1e-4
        for k in nb_probs.keys()
    )

    match = pred_match and prob_match
    if not match:
        all_match = False

    match_str = 'PASS' if match else 'FAIL'
    display_text = sample[:40] + ('...' if len(sample) > 40 else '')
    print(f'  {i+1}. {display_text:<43s} {nb_label:<12s} {api_label:<12s} {match_str:<6s}')

    results_table.append({
        'sample': sample,
        'notebook_pred': nb_pred,
        'notebook_label': nb_label,
        'api_pred': api_pred,
        'api_label': api_label,
        'nb_probs': nb_probs,
        'api_probs': api_probs,
        'match': match,
    })

print('-' * 70)
if all_match:
    print(f'✓ ALL {len(comparison_samples)} SAMPLES MATCH — Notebook == FastAPI')
else:
    print(f'✗ SOME SAMPLES DO NOT MATCH — investigate above')
print('=' * 70)

NOTEBOOK vs FASTAPI COMPARISON
Sample                                        Notebook     FastAPI      Match 
----------------------------------------------------------------------
  1. هذا المنتج ممتاز جدا                        Positive (1) Positive (1) PASS  
  2. المنتج سيء جدا ولا أنصح به                  Negative (0) Negative (0) PASS  
  3. جودة عالية وشحن سريع وخدمة ممتازة           Positive (1) Positive (1) PASS  
  4. خدمة سيئة جدا وانتظار طويل ولا أنصح         Negative (0) Negative (0) PASS  
  5. أحب هذا المتجر ومنتجاتهم ممتازة جدا         Positive (1) Positive (1) PASS  
  6. منتج عادي لا يستحق السعر المطلوب            Negative (0) Negative (0) PASS  
  7. هذا أفضل منتج استخدمته هذا العام            Negative (0) Negative (0) PASS  
  8. المنتج وصل تالف ومكسور                      Negative (0) Negative (0) PASS  
----------------------------------------------------------------------
✓ ALL 8 SAMPLES MATCH — Notebook == FastAPI


---

## 14. Programmatic API Testing

Test the full HTTP stack: Request → FastAPI → Pydantic → Preprocessing → Model → Response

In [10]:
print('=' * 70)
print('PROGRAMMATIC API TEST')
print('=' * 70)

# Test root endpoint
resp = client.get('/')
assert resp.status_code == 200
root_data = resp.json()
print(f'  GET /  → status={root_data["status"]}, model={root_data["model_type"]}')

# Test health endpoint
resp = client.get('/health')
assert resp.status_code == 200
print(f'  GET /health → status={resp.json()["status"]}')

# Test predict endpoint with timing
t0 = time.time()
resp = client.post('/predict', json={'text': 'هذا المنتج ممتاز جدا'})
t_total = (time.time() - t0) * 1000
assert resp.status_code == 200
pred_data = resp.json()
print(f'  POST /predict → label={pred_data["label"]}, confidence={pred_data["confidence"]}, time={t_total:.1f}ms')
print(f'    probabilities: {pred_data["probabilities"]}')
print(f'    model_version: {pred_data["model_version"]}')

# Test response model validation
validated = PredictionResponse(**pred_data)
print(f'  Response model validation: OK')

print(f'\n✓ Full API stack test passed.')
print('=' * 70)

PROGRAMMATIC API TEST
  GET /  → status=ok, model=LogisticRegression
  GET /health → status=healthy
  POST /predict → label=Positive (1), confidence=0.9761, time=3.6ms
    probabilities: {'Negative (0)': 0.0239, 'Positive (1)': 0.9761}
    model_version: 1.0.0
  Response model validation: OK

✓ Full API stack test passed.


---

## 15. Final Validation Summary

In [11]:
print('=' * 70)
print('DAY 2 FINAL VALIDATION')
print('=' * 70)

checks = []

# 1. Artifacts load
try:
    artifacts = load_artifacts()
    checks.append(('Artifacts load successfully', 'PASS'))
except Exception as e:
    checks.append(('Artifacts load successfully', f'FAIL: {e}'))

# 2. Model type correct
try:
    assert type(artifacts['model']).__name__ == 'LogisticRegression'
    checks.append(('Model type correct (LogisticRegression)', 'PASS'))
except Exception as e:
    checks.append(('Model type correct (LogisticRegression)', f'FAIL: {e}'))

# 3. Vectorizer vocab size
try:
    assert len(artifacts['vectorizer'].vocabulary_) == 10000
    checks.append(('Vectorizer vocab = 10,000', 'PASS'))
except Exception as e:
    checks.append(('Vectorizer vocab = 10,000', f'FAIL: {e}'))

# 4. Lemma table loaded
try:
    assert len(artifacts['lemma_table']) > 10000
    checks.append((f'Lemma table loaded ({len(artifacts["lemma_table"]):,} entries)', 'PASS'))
except Exception as e:
    checks.append((f'Lemma table loaded', f'FAIL: {e}'))

# 5. FastAPI app created
try:
    assert app is not None
    checks.append(('FastAPI app created', 'PASS'))
except Exception as e:
    checks.append(('FastAPI app created', f'FAIL: {e}'))

# 6. Root endpoint
try:
    resp = client.get('/')
    assert resp.status_code == 200
    checks.append(('GET / endpoint works', 'PASS'))
except Exception as e:
    checks.append(('GET / endpoint works', f'FAIL: {e}'))

# 7. Health endpoint
try:
    resp = client.get('/health')
    assert resp.status_code == 200
    checks.append(('GET /health endpoint works', 'PASS'))
except Exception as e:
    checks.append(('GET /health endpoint works', f'FAIL: {e}'))

# 8. Predict endpoint - positive
try:
    resp = client.post('/predict', json={'text': 'هذا المنتج ممتاز جدا'})
    assert resp.status_code == 200
    assert resp.json()['prediction'] == 1
    checks.append(('POST /predict (positive input)', 'PASS'))
except Exception as e:
    checks.append(('POST /predict (positive input)', f'FAIL: {e}'))

# 9. Predict endpoint - negative
try:
    resp = client.post('/predict', json={'text': 'المنتج سيء جدا'})
    assert resp.status_code == 200
    assert resp.json()['prediction'] == 0
    checks.append(('POST /predict (negative input)', 'PASS'))
except Exception as e:
    checks.append(('POST /predict (negative input)', f'FAIL: {e}'))

# 10. Pydantic validation - empty text
try:
    resp = client.post('/predict', json={'text': ''})
    assert resp.status_code == 422
    checks.append(('Pydantic rejects empty text', 'PASS'))
except Exception as e:
    checks.append(('Pydantic rejects empty text', f'FAIL: {e}'))

# 11. Pydantic validation - missing field
try:
    resp = client.post('/predict', json={})
    assert resp.status_code == 422
    checks.append(('Pydantic rejects missing field', 'PASS'))
except Exception as e:
    checks.append(('Pydantic rejects missing field', f'FAIL: {e}'))

# 12. Pydantic validation - wrong type
try:
    resp = client.post('/predict', json={'text': 12345})
    assert resp.status_code == 422
    checks.append(('Pydantic rejects wrong type', 'PASS'))
except Exception as e:
    checks.append(('Pydantic rejects wrong type', f'FAIL: {e}'))

# 13. Notebook vs API consistency
try:
    nb_pred, _ = notebook_predict('هذا المنتج ممتاز جدا')
    api_data = api_predict('هذا المنتج ممتاز جدا')
    assert nb_pred == api_data['prediction']
    checks.append(('Notebook == API prediction', 'PASS'))
except Exception as e:
    checks.append(('Notebook == API prediction', f'FAIL: {e}'))

# 14. Preprocessing consistency (same functions)
try:
    test_text = 'هذا المنتج ممتاز جدا'
    nb_cleaned = preprocess_text(test_text, lemma_table=lemma_table)
    api_cleaned = preprocess_text(test_text, lemma_table=artifacts['lemma_table'])
    assert nb_cleaned == api_cleaned
    checks.append(('Preprocessing consistency (same functions)', 'PASS'))
except Exception as e:
    checks.append(('Preprocessing consistency (same functions)', f'FAIL: {e}'))

# 15. No model retraining
try:
    # Verify we loaded from disk, not retrained
    model_file = get_artifacts_dir() / 'model.joblib'
    vec_file = get_artifacts_dir() / 'vectorizer.joblib'
    assert model_file.exists()
    assert vec_file.exists()
    checks.append(('Artifacts loaded from disk (no retraining)', 'PASS'))
except Exception as e:
    checks.append(('Artifacts loaded from disk (no retraining)', f'FAIL: {e}'))

# Print summary
print()
pass_count = 0
fail_count = 0
for name, status in checks:
    icon = '✅' if status == 'PASS' else '❌'
    print(f'  {icon} {name:<50s} {status}')
    if status == 'PASS':
        pass_count += 1
    else:
        fail_count += 1

print(f'\n  {pass_count}/{len(checks)} checks PASS')
if fail_count > 0:
    print(f'  {fail_count} CHECKS FAILED')
else:
    print(f'\n  ✓ ALL CHECKS PASSED')
print('=' * 70)

DAY 2 FINAL VALIDATION

  ✅ Artifacts load successfully                        PASS
  ✅ Model type correct (LogisticRegression)            PASS
  ✅ Vectorizer vocab = 10,000                          PASS
  ✅ Lemma table loaded (20,381 entries)                PASS
  ✅ FastAPI app created                                PASS
  ✅ GET / endpoint works                               PASS
  ✅ GET /health endpoint works                         PASS
  ✅ POST /predict (positive input)                     PASS
  ✅ POST /predict (negative input)                     PASS
  ✅ Pydantic rejects empty text                        PASS
  ✅ Pydantic rejects missing field                     PASS
  ✅ Pydantic rejects wrong type                        PASS
  ✅ Notebook == API prediction                         PASS
  ✅ Preprocessing consistency (same functions)         PASS
  ✅ Artifacts loaded from disk (no retraining)         PASS

  15/15 checks PASS

  ✓ ALL CHECKS PASSED


---

## 16. Multi-Sample Comparison Table

Full comparison table with probability details for all test samples.

In [12]:
print('=' * 90)
print('MULTI-SAMPLE COMPARISON TABLE')
print('=' * 90)

print(f'{"#":<3s} {"Sample":<35s} {"NB Label":<12s} {"API Label":<12s} '
      f'{"NB P(+)":<10s} {"API P(+)":<10s} {"Match":<6s}')
print('-' * 90)

for i, r in enumerate(results_table):
    sample_display = r['sample'][:32] + ('...' if len(r['sample']) > 32 else '')
    nb_p_pos = r['nb_probs'].get('Positive (1)', 0.0)
    api_p_pos = r['api_probs'].get('Positive (1)', 0.0)
    match_str = 'PASS' if r['match'] else 'FAIL'
    print(f'{i+1:<3d} {sample_display:<35s} {r["notebook_label"]:<12s} {r["api_label"]:<12s} '
          f'{nb_p_pos:<10.4f} {api_p_pos:<10.4f} {match_str:<6s}')

print('-' * 90)
total = len(results_table)
passed = sum(1 for r in results_table if r['match'])
print(f'Total: {total} samples, {passed} PASS, {total - passed} FAIL')
if passed == total:
    print('✓ ALL SAMPLES MATCH — Notebook == FastAPI')
print('=' * 90)

MULTI-SAMPLE COMPARISON TABLE
#   Sample                              NB Label     API Label    NB P(+)    API P(+)   Match 
------------------------------------------------------------------------------------------
1   هذا المنتج ممتاز جدا                Positive (1) Positive (1) 0.9761     0.9761     PASS  
2   المنتج سيء جدا ولا أنصح به          Negative (0) Negative (0) 0.4937     0.4937     PASS  
3   جودة عالية وشحن سريع وخدمة ممتاز... Positive (1) Positive (1) 0.9410     0.9410     PASS  
4   خدمة سيئة جدا وانتظار طويل ولا أ... Negative (0) Negative (0) 0.3275     0.3275     PASS  
5   أحب هذا المتجر ومنتجاتهم ممتازة ... Positive (1) Positive (1) 0.9678     0.9678     PASS  
6   منتج عادي لا يستحق السعر المطلوب    Negative (0) Negative (0) 0.4239     0.4239     PASS  
7   هذا أفضل منتج استخدمته هذا العام    Negative (0) Negative (0) 0.4642     0.4642     PASS  
8   المنتج وصل تالف ومكسور              Negative (0) Negative (0) 0.3837     0.3837     PASS  
------------------------

---

## 17. Day 3 Handoff

The Streamlit dashboard (Day 3) should interact with the FastAPI service as follows:

### FastAPI Application

| Property | Value |
|:---------|:------|
| **File** | `BinX_Week_09/Day2/main.py` |
| **Start command** | `uvicorn main:app --reload` (from Day2 directory) |
| **Port** | 8000 |
| **Docs URL** | `http://127.0.0.1:8000/docs` |

### Artifact Locations

| Artifact | Path |
|:---------|:-----|
| Model | `BinX_Week_09/Day1/artifacts/model.joblib` |
| Vectorizer | `BinX_Week_09/Day1/artifacts/vectorizer.joblib` |
| Lemma table | `BinX_Week_09/Day1/artifacts/lemma_table.json` |
| Config | `BinX_Week_09/Day1/artifacts/preprocessing_config.json` |

### API Contract

#### Request
```json
POST /predict
{"text": "Arabic review text"}
```

#### Response
```json
{
    "prediction": 0 or 1,
    "label": "Negative (0)" or "Positive (1)",
    "confidence": 0.95,
    "probabilities": {"Negative (0)": 0.05, "Positive (1)": 0.95},
    "model_version": "1.0.0",
    "preprocessing_steps": 9
}
```

### Integration Options for Day 3

1. **HTTP calls**: Streamlit calls `http://127.0.0.1:8000/predict` via `requests`
2. **Direct reuse**: Streamlit imports `preprocessing.py` + loads artifacts directly

> **Recommendation**: Use option 1 (HTTP calls) so the Streamlit app tests the actual
> API endpoint, reinforcing the Day 2 serving pipeline.

### Validation Result

- All Day 2 checks: **PASS**
- Notebook vs API consistency: **verified**
- Pydantic validation: **working**
- Ready for Day 3: **yes**